In [ ]:
# pip installs

!pip install -q datasets peft requests torch bitsandbytes transformers trl accelerate sentencepiece matplotlib
#duc

Đoạn code này cài đặt các thư viện Python cần thiết cho một dự án học máy (machine learning) hoặc xử lý ngôn ngữ tự nhiên (NLP). Cụ thể:

datasets: Thư viện của Hugging Face để tải và xử lý dữ liệu cho các mô hình học máy.

peft: Một thư viện có thể liên quan đến phương pháp học nâng cao, thường được sử dụng trong các mô hình học sâu.

requests: Thư viện để gửi các yêu cầu HTTP (ví dụ: tải dữ liệu từ các API).

torch: Thư viện PyTorch, được sử dụng để xây dựng và huấn luyện các mô hình học sâu.

bitsandbytes: Thư viện tối ưu hóa cho các mô hình học sâu, giúp giảm bớt chi phí tính toán.

transformers: Thư viện từ Hugging Face, chứa các mô hình NLP tiên tiến (như BERT, GPT).

trl: Thư viện để hỗ trợ các mô hình chuyển giao học (transfer learning).

accelerate: Thư viện của Hugging Face giúp tăng tốc huấn luyện mô hình.

sentencepiece: Thư viện cho phép mã hóa văn bản thành các phần tử nhỏ hơn (subwords), thường dùng trong các mô hình NLP.

matplotlib: Thư viện vẽ đồ thị và hình ảnh trong Python, thường dùng để trực quan hóa dữ liệu hoặc kết quả mô hình.

In [ ]:
# imports

import os
import re
import math
from tqdm import tqdm
from google.colab import userdata
from huggingface_hub import login
import torch
import torch.nn.functional as F
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from datasets import load_dataset, Dataset, DatasetDict
from datetime import datetime
from peft import PeftModel
import matplotlib.pyplot as plt

In [ ]:
# Constants

BASE_MODEL = "meta-llama/Meta-Llama-3.1-8B"
PROJECT_NAME = "pricer"
HF_USER = "ed-donner" # your HF name here! Or use mine if you just want to reproduce my results.

# The run itself

RUN_NAME = "2024-09-13_13.04.39"
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
REVISION = "e8d637df551603dc86cd7a1598a8f44af4d7ae36" # or REVISION = None
FINETUNED_MODEL = f"{HF_USER}/{PROJECT_RUN_NAME}"

# Uncomment this line if you wish to use my model
# FINETUNED_MODEL = f"ed-donner/{PROJECT_RUN_NAME}"

# Data

DATASET_NAME = f"{HF_USER}/pricer-data"
# Or just use the one I've uploaded
# DATASET_NAME = "ed-donner/pricer-data"

# Hyperparameters for QLoRA

QUANT_4_BIT = True

%matplotlib inline

# Used for writing to output in color

GREEN = "\033[92m"
YELLOW = "\033[93m"
RED = "\033[91m"
RESET = "\033[0m"
COLOR_MAP = {"red":RED, "orange": YELLOW, "green": GREEN}

Constants (Hằng số):

BASE_MODEL: Xác định tên mô hình cơ sở, ở đây là "meta-llama/Meta-Llama-3.1-8B".

PROJECT_NAME: Tên dự án, trong trường hợp này là "pricer".

HF_USER: Tên người dùng Hugging Face của bạn, dùng để xác định bạn là ai khi lưu trữ và chia sẻ mô hình.

RUN_NAME: Tên của lần chạy hiện tại (thường bao gồm thời gian và một số thông tin cụ thể).

PROJECT_RUN_NAME: Tên của lần chạy kết hợp giữa tên dự án và tên lần chạy.

REVISION: ID sửa đổi hoặc phiên bản mã nguồn của dự án. Để trống nếu không cần chỉ định.

FINETUNED_MODEL: Địa chỉ mô hình đã được tinh chỉnh trên Hugging Face.

DATASET_NAME: Địa chỉ bộ dữ liệu (dataset) trên Hugging Face.

Hyperparameters for QLoRA:

QUANT_4_BIT: Biến xác định liệu có sử dụng QLoRA (quantized LoRA) với 4 bit hay không. Đây là một kỹ thuật để tối ưu hóa bộ nhớ khi huấn luyện mô hình.

Visualization:

%matplotlib inline: Lệnh này là của Jupyter Notebook, cho phép vẽ đồ thị ngay trong notebook mà không cần mở cửa sổ mới.

Color output (Màu sắc đầu ra):

Các biến GREEN, YELLOW, RED, và RESET được sử dụng để tô màu cho các thông báo khi in ra màn hình (thường để làm nổi bật các thông báo quan trọng hoặc lỗi).

COLOR_MAP: Tạo một từ điển ánh xạ các màu sắc đến các biến màu đã định nghĩa trước đó.

Tóm tắt:
Đoạn code này chủ yếu thiết lập môi trường cho một dự án huấn luyện mô hình, bao gồm việc xác định mô hình cơ sở, tên dự án, các thông số huấn luyện, và cấu hình hiển thị đầu ra trong màu sắc.

In [ ]:
# Log in to HuggingFace

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

Đoạn code này thực hiện đăng nhập vào Hugging Face bằng token của người dùng:

hf_token = userdata.get('HF_TOKEN'):
Đoạn mã này lấy giá trị của token Hugging Face từ một đối tượng userdata. Token này cần được tạo từ tài khoản Hugging Face và có quyền truy cập vào tài nguyên của bạn (mô hình, dữ liệu, v.v.).

login(hf_token, add_to_git_credential=True):
Lệnh này đăng nhập vào Hugging Face bằng token hf_token. Thêm add_to_git_credential=True giúp lưu trữ token này vào bộ nhớ Git, giúp bạn dễ dàng sử dụng trong các thao tác Git sau này (ví dụ: khi đẩy mô hình lên Hugging Face).

Tóm tắt:
Mục đích của đoạn mã là đăng nhập vào Hugging Face bằng cách sử dụng token cá nhân, giúp bạn có thể tải lên hoặc tải xuống mô hình và dữ liệu từ tài khoản Hugging Face của mình.

In [ ]:
dataset = load_dataset(DATASET_NAME)
train = dataset['train']
test = dataset['test']

In [ ]:
test[0]

## Now load the Tokenizer and Model

In [ ]:
# pick the right quantization (thank you Robert M. for spotting the bug with the 8 bit version!)

if QUANT_4_BIT:
  quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
  )
else:
  quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.bfloat16
  )

if QUANT_4_BIT::

Kiểm tra xem biến QUANT_4_BIT có được đặt là True không. Nếu True, sử dụng cấu hình cho 4-bit quantization. Nếu không, sẽ dùng cấu hình cho 8-bit quantization.

Khi QUANT_4_BIT là True:

BitsAndBytesConfig(...): Đây là cách cấu hình cho việc tải mô hình ở định dạng 4-bit.

load_in_4bit=True: Chỉ định rằng mô hình sẽ được tải vào với độ chính xác 4-bit.

bnb_4bit_use_double_quant=True: Kích hoạt việc sử dụng "double quantization" để giảm mất mát thông tin trong quá trình lượng tử hóa.

bnb_4bit_compute_dtype=torch.bfloat16: Chỉ định kiểu dữ liệu cho phép tính toán, ở đây sử dụng torch.bfloat16 (loại dữ liệu 16-bit floating point được tối ưu cho các mô hình học sâu).

bnb_4bit_quant_type="nf4": Chỉ định loại lượng tử hóa, ở đây là "nf4" (một dạng quantization tối ưu cho 4-bit).

Khi QUANT_4_BIT là False:

BitsAndBytesConfig(...): Cấu hình cho 8-bit quantization.

load_in_8bit=True: Chỉ định rằng mô hình sẽ được tải vào với độ chính xác 8-bit.

bnb_8bit_compute_dtype=torch.bfloat16: Cũng sử dụng torch.bfloat16 cho phép tính toán với 8-bit.

Tóm tắt:
Đoạn mã này giúp lựa chọn cấu hình lượng tử hóa phù hợp (4-bit hoặc 8-bit) cho mô hình, tùy thuộc vào việc bạn đã chọn QUANT_4_BIT hay chưa. Cấu hình này giúp tối ưu hóa bộ nhớ và tính toán khi làm việc với các mô hình học sâu.

In [ ]:
# Load the Tokenizer and the Model

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

# Load the fine-tuned model with PEFT
if REVISION:
  fine_tuned_model = PeftModel.from_pretrained(base_model, FINETUNED_MODEL, revision=REVISION)
else:
  fine_tuned_model = PeftModel.from_pretrained(base_model, FINETUNED_MODEL)


print(f"Memory footprint: {fine_tuned_model.get_memory_footprint() / 1e6:.1f} MB")

Tải Tokenizer:

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True):

Dùng AutoTokenizer từ Hugging Face để tải tokenizer cho mô hình cơ sở (được chỉ định trong BASE_MODEL).

trust_remote_code=True cho phép tải mã từ mô hình từ xa (có thể là mô hình của một người khác).

tokenizer.pad_token = tokenizer.eos_token:

Đặt token padding (pad_token) giống với token kết thúc chuỗi (eos_token), đảm bảo khi mô hình gặp độ dài không đều của các chuỗi, nó sẽ sử dụng token kết thúc để lấp đầy.

tokenizer.padding_side = "right":

Xác định rằng token padding sẽ được thêm vào bên phải của chuỗi văn bản (thay vì bên trái).

Tải Mô hình Cơ sở:

base_model = AutoModelForCausalLM.from_pretrained(...):

Dùng AutoModelForCausalLM để tải mô hình ngôn ngữ có thể sinh văn bản (causal language model) từ Hugging Face.

quantization_config=quant_config: Chuyển cấu hình lượng tử hóa (quant_config) đã thiết lập trước đó.

device_map="auto": Mô hình sẽ được tự động phân bổ vào các thiết bị (CPU/GPU) có sẵn.

base_model.generation_config.pad_token_id = tokenizer.pad_token_id:

Thiết lập pad_token_id của mô hình cho phù hợp với pad_token_id của tokenizer.

Tải Mô hình Đã Tinh Chỉnh (Fine-Tuned):

Kiểm tra nếu có REVISION (phiên bản sửa đổi) được cung cấp:

Nếu có, sử dụng PeftModel.from_pretrained(base_model, FINETUNED_MODEL, revision=REVISION) để tải mô hình đã tinh chỉnh với phiên bản sửa đổi.

Nếu không có, sử dụng PeftModel.from_pretrained(base_model, FINETUNED_MODEL) để tải mô hình đã tinh chỉnh mà không có phiên bản cụ thể.

PeftModel là mô hình đã được tinh chỉnh sử dụng phương pháp Progressive Embedding Finetuning (PEFT).

In Memory Footprint:

print(f"Memory footprint: {fine_tuned_model.get_memory_footprint() / 1e6:.1f} MB"):

In ra dung lượng bộ nhớ mà mô hình đã tinh chỉnh (fine_tuned_model) chiếm, được tính bằng MB.

Tóm tắt:
Đoạn code này tải mô hình ngôn ngữ cơ sở từ Hugging Face, cấu hình tokenizer và mô hình với các thiết lập cần thiết, sau đó tải mô hình đã tinh chỉnh nếu có, và in ra dung lượng bộ nhớ mà mô hình đã chiếm dụng. Việc sử dụng PEFT giúp tối ưu quá trình tinh chỉnh mô hình mà không làm mất quá nhiều bộ nhớ.

In [ ]:
fine_tuned_model

In [ ]:
def extract_price(s):
    if "Price is $" in s:
      contents = s.split("Price is $")[1]
      contents = contents.replace(',','')
      match = re.search(r"[-+]?\d*\.\d+|\d+", contents)
      return float(match.group()) if match else 0
    return 0

1.Kiểm tra chuỗi có chứa cụm "Price is $":
Hàm kiểm tra xem chuỗi s có chứa cụm "Price is $" hay không. Nếu có, tiếp tục xử lý để trích xuất giá.
2.Tách chuỗi sau cụm "Price is $":
Dùng phương thức .split("Price is $") để chia chuỗi tại vị trí "Price is $". Phần sau dấu $ (tức là giá trị) sẽ nằm ở chỉ mục 1 của danh sách kết quả từ phép tách.
3.Loại bỏ dấu phẩy trong chuỗi:
Nếu giá trị chứa dấu phẩy (ví dụ: "1,000.50"), hàm sẽ loại bỏ dấu phẩy để giá trị trở thành một số hợp lệ cho phép chuyển đổi.
Sử dụng biểu thức chính quy (regex) để tìm số trong chuỗi contents. Biểu thức này tìm kiếm một số thực hoặc số nguyên.

[-+]?: Tùy chọn dấu cộng (+) hoặc dấu trừ (-) ở đầu số.

\d*\.\d+: Tìm số thực (có dấu chấm).

\d+: Tìm số nguyên.
Nếu tìm thấy số (match), chuyển nó thành số thực (float) và trả về.

Nếu không tìm thấy số, trả về 0.
Nếu chuỗi không chứa cụm "Price is $", hàm trả về 0.

In [ ]:
extract_price("Price is $a fabulous 899.99 or so")

Kết quả:
Hàm sẽ trả về giá trị 899.99 dưới dạng số thực (float).

In [ ]:
# Original prediction function takes the most likely next token

def model_predict(prompt):
    set_seed(42)
    inputs = tokenizer.encode(prompt, return_tensors="pt").to("cuda")
    attention_mask = torch.ones(inputs.shape, device="cuda")
    outputs = fine_tuned_model.generate(inputs, attention_mask=attention_mask, max_new_tokens=3, num_return_sequences=1)
    response = tokenizer.decode(outputs[0])
    return extract_price(response)

óm tắt:
Hàm model_predict(prompt) sẽ:

Nhận vào một câu lệnh (prompt).

Mã hóa câu lệnh và sử dụng mô hình đã được tinh chỉnh để sinh ra văn bản tiếp theo.

Giải mã phản hồi từ mô hình thành văn bản.

Cuối cùng, hàm sẽ sử dụng extract_price để trích xuất giá trị từ phản hồi và trả về kết quả đó.

In [ ]:
# An improved prediction function takes a weighted average of the top 3 choices
# This code would be more complex if we couldn't take advantage of the fact
# That Llama generates 1 token for any 3 digit number

top_K = 3

def improved_model_predict(prompt, device="cuda"):
    set_seed(42)
    inputs = tokenizer.encode(prompt, return_tensors="pt").to(device)
    attention_mask = torch.ones(inputs.shape, device=device)

    with torch.no_grad():
        outputs = fine_tuned_model(inputs, attention_mask=attention_mask)
        next_token_logits = outputs.logits[:, -1, :].to('cpu')

    next_token_probs = F.softmax(next_token_logits, dim=-1)
    top_prob, top_token_id = next_token_probs.topk(top_K)
    prices, weights = [], []
    for i in range(top_K):
      predicted_token = tokenizer.decode(top_token_id[0][i])
      probability = top_prob[0][i]
      try:
        result = float(predicted_token)
      except ValueError as e:
        result = 0.0
      if result > 0:
        prices.append(result)
        weights.append(probability)
    if not prices:
      return 0.0, 0.0
    total = sum(weights)
    weighted_prices = [price * weight / total for price, weight in zip(prices, weights)]
    return sum(weighted_prices).item()

Hàm improved_model_predict(prompt) sẽ:

Lấy các token tiếp theo từ mô hình với xác suất cao nhất (top_K lựa chọn).

Chuyển các token này thành giá trị số, sau đó tính toán giá trị có trọng số dựa trên xác suất của mỗi token.

Trả về giá trị dự đoán cuối cùng, tính theo trung bình có trọng số.improved_model_predict("The price of the item is")
Mô hình sẽ chọn ba lựa chọn token có xác suất cao nhất, ví dụ: "899.99", "1200.50", "950.00", sau đó tính toán giá trị có trọng số của chúng và trả về kết quả trung bình, ví dụ: 950.33.

In [ ]:
class Tester:

    def __init__(self, predictor, data, title=None, size=250):
        self.predictor = predictor
        self.data = data
        self.title = title or predictor.__name__.replace("_", " ").title()
        self.size = size
        self.guesses = []
        self.truths = []
        self.errors = []
        self.sles = []
        self.colors = []

    def color_for(self, error, truth):
        if error<40 or error/truth < 0.2:
            return "green"
        elif error<80 or error/truth < 0.4:
            return "orange"
        else:
            return "red"

    def run_datapoint(self, i):
        datapoint = self.data[i]
        guess = self.predictor(datapoint["text"])
        truth = datapoint["price"]
        error = abs(guess - truth)
        log_error = math.log(truth+1) - math.log(guess+1)
        sle = log_error ** 2
        color = self.color_for(error, truth)
        title = datapoint["text"].split("\n\n")[1][:20] + "..."
        self.guesses.append(guess)
        self.truths.append(truth)
        self.errors.append(error)
        self.sles.append(sle)
        self.colors.append(color)
        print(f"{COLOR_MAP[color]}{i+1}: Guess: ${guess:,.2f} Truth: ${truth:,.2f} Error: ${error:,.2f} SLE: {sle:,.2f} Item: {title}{RESET}")

    def chart(self, title):
        max_error = max(self.errors)
        plt.figure(figsize=(12, 8))
        max_val = max(max(self.truths), max(self.guesses))
        plt.plot([0, max_val], [0, max_val], color='deepskyblue', lw=2, alpha=0.6)
        plt.scatter(self.truths, self.guesses, s=3, c=self.colors)
        plt.xlabel('Ground Truth')
        plt.ylabel('Model Estimate')
        plt.xlim(0, max_val)
        plt.ylim(0, max_val)
        plt.title(title)
        plt.show()

    def report(self):
        average_error = sum(self.errors) / self.size
        rmsle = math.sqrt(sum(self.sles) / self.size)
        hits = sum(1 for color in self.colors if color=="green")
        title = f"{self.title} Error=${average_error:,.2f} RMSLE={rmsle:,.2f} Hits={hits/self.size*100:.1f}%"
        self.chart(title)

    def run(self):
        self.error = 0
        for i in range(self.size):
            self.run_datapoint(i)
        self.report()

    @classmethod
    def test(cls, function, data):
        cls(function, data).run()

1. Khởi tạo (__init__)
self.predictor: Hàm dự đoán mà bạn sẽ sử dụng (chẳng hạn như model_predict hoặc improved_model_predict).

self.data: Dữ liệu để kiểm tra, giả sử là danh sách các từ điển chứa "text" (đầu vào) và "price" (giá trị thực).

self.title: Tiêu đề cho bài kiểm tra, mặc định là tên hàm dự đoán chuyển thành dạng tiêu đề (có thể tuỳ chỉnh).

self.size: Số lượng điểm dữ liệu cần kiểm tra (mặc định là 250).

Các danh sách (self.guesses, self.truths, self.errors, self.sles, self.colors) dùng để lưu trữ kết quả của các phép tính trong quá trình chạy mô hình.

2. Phương thức color_for()\
Dựa trên lỗi tuyệt đối (error) và tỷ lệ lỗi so với giá trị thực (truth), phương thức này gán một màu cho kết quả:

Xanh (green): Lỗi nhỏ hơn 40 hoặc tỷ lệ lỗi nhỏ hơn 20% so với giá trị thực.

Cam (orange): Lỗi nhỏ hơn 80 hoặc tỷ lệ lỗi nhỏ hơn 40%.

Đỏ (red): Các trường hợp còn lại.

3. Phương thức run_datapoint()
Dựa trên lỗi tuyệt đối (error) và tỷ lệ lỗi so với giá trị thực (truth), phương thức này gán một màu cho kết quả:

Xanh (green): Lỗi nhỏ hơn 40 hoặc tỷ lệ lỗi nhỏ hơn 20% so với giá trị thực.

Cam (orange): Lỗi nhỏ hơn 80 hoặc tỷ lệ lỗi nhỏ hơn 40%.

Đỏ (red): Các trường hợp còn lại.


4. Phương thức chart()
Vẽ một biểu đồ phân tán (scatter plot) giữa giá trị thực và giá trị dự đoán.

Biểu đồ có màu sắc của các điểm dữ liệu để dễ dàng phân biệt các mức độ lỗi (green, orange, red).

Biểu đồ cũng vẽ một đường chéo màu deepskyblue để so sánh sự tương đồng giữa giá trị thực và dự đoán.


5. Phương thức report()
Tính toán và in ra các chỉ số đánh giá:

Lỗi trung bình (average error): Trung bình của tất cả các lỗi tuyệt đối.

RMSLE: Căn bậc hai của trung bình các lỗi logarithmic bình phương.

Hit rate: Tỷ lệ các điểm dữ liệu có màu green (lỗi nhỏ).

Gọi phương thức chart() để vẽ biểu đồ với tiêu đề chứa các thông tin đánh giá.


6. Phương thức run()
Lặp qua các điểm dữ liệu và chạy hàm run_datapoint() cho mỗi điểm dữ liệu.

Sau khi chạy xong tất cả các điểm dữ liệu, gọi report() để xuất báo cáo và biểu đồ.


7. Phương thức lớp test()
Phương thức này là một phương thức lớp, cho phép bạn dễ dàng kiểm tra một hàm dự đoán (function) trên một tập dữ liệu (data).

Nó tạo một đối tượng Tester và gọi phương thức run() để thực hiện đánh giá.

Tóm tắt:
Lớp Tester giúp kiểm tra và đánh giá một hàm dự đoán bằng cách:

Chạy mô hình dự đoán trên từng điểm dữ liệu.

Tính toán các chỉ số lỗi như lỗi tuyệt đối, RMSLE, và hit rate.

Vẽ biểu đồ phân tán so sánh giữa giá trị thực và giá trị dự đoán.

In báo cáo chi tiết về hiệu suất mô hình.










In [ ]:
Tester.test(improved_model_predict, test)